In [1]:
!pip install torch_geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.9 MB/s eta 0:00:00


In [2]:
import torch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F

In [3]:
import bz2
import os

compressed_file_path = 'causal_dataset.pt.bz2'
decompressed_file_path = 'causal_dataset.pt'

if not os.path.exists(decompressed_file_path):
    print(f"Decompressing {compressed_file_path}...")
    with bz2.open(compressed_file_path, 'rb') as f_in:
        with open(decompressed_file_path, 'wb') as f_out:
            for data in iter(lambda: f_in.read(100 * 1024), b''):
                f_out.write(data)
    print(f"Decompression complete. File saved as {decompressed_file_path}")
else:
    print(f"{decompressed_file_path} already exists. Skipping decompression.")

causal_dataset.pt already exists. Skipping decompression.


In [4]:
import random
import torch
from torch_geometric.loader import DataLoader

random.seed(42)

dataset = torch.load('causal_dataset.pt', weights_only=False)

pos = [d for d in dataset if d.y.item() == 1]
neg = [d for d in dataset if d.y.item() == 0]
random.shuffle(pos)
random.shuffle(neg)
print(f"Positives: {len(pos)}, Negatives: {len(neg)}")

# Split each class 80/20 separately to avoid leakage
pos_split, neg_split = int(0.8*len(pos)), int(0.8*len(neg))
train_pos, test_pos = pos[:pos_split], pos[pos_split:]
train_neg, test_neg = neg[:neg_split], neg[neg_split:]

train_dataset = train_pos + train_neg
random.shuffle(train_dataset)

# Keep test set at NATURAL ratio
test_dataset = test_pos + test_neg

print(f"Train: {len(train_dataset)} ({len(train_pos)} pos / {len(train_neg)} neg)")
print(f"Test: {len(test_dataset)} ({len(test_pos)} pos / {len(test_neg)} neg)")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

Positives: 6002, Negatives: 42876
Train: 39101 (4801 pos / 34300 neg)
Test: 9777 (1201 pos / 8576 neg)


In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

class CausalGNN(torch.nn.Module):
    # Bumped hidden channels to 64 to give the model a bigger brain to learn physics
    def __init__(self, in_channels=17, hidden=64):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = torch.nn.Linear(hidden, 1)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.lin(

num_negatives = len(train_neg)
num_positives = len(train_pos)
pos_weight_val = num_negatives / num_positives
pos_weight = torch.tensor([pos_weight_val])

model = CausalGNN()

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)

for epoch in range(50):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)

        loss = F.binary_cross_entropy_with_logits(
            out.view(-1),
            batch.y,
            pos_weight=pos_weight.to(out.device)
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch}: loss={total_loss/len(train_loader):.4f}")

Epoch 0: loss=0.9830
Epoch 1: loss=0.9208
Epoch 2: loss=0.8928
Epoch 3: loss=0.8853
Epoch 4: loss=0.8933
Epoch 5: loss=0.8830
Epoch 6: loss=0.8741
Epoch 7: loss=0.8687
Epoch 8: loss=0.8658
Epoch 9: loss=0.8782
Epoch 10: loss=0.8653
Epoch 11: loss=0.8556
Epoch 12: loss=0.8676
Epoch 13: loss=0.8625
Epoch 14: loss=0.8603
Epoch 15: loss=0.8546
Epoch 16: loss=0.8493
Epoch 17: loss=0.8571
Epoch 18: loss=0.8490
Epoch 19: loss=0.8461
Epoch 20: loss=0.8486
Epoch 21: loss=0.8456
Epoch 22: loss=0.8609
Epoch 23: loss=0.8638
Epoch 24: loss=0.8620
Epoch 25: loss=0.8440
Epoch 26: loss=0.8527
Epoch 27: loss=0.8391
Epoch 28: loss=0.8511
Epoch 29: loss=0.8420
Epoch 30: loss=0.8560
Epoch 31: loss=0.8389
Epoch 32: loss=0.8719
Epoch 33: loss=0.8381
Epoch 34: loss=0.8761
Epoch 35: loss=0.8506
Epoch 36: loss=0.8509
Epoch 37: loss=0.8604
Epoch 38: loss=0.8534
Epoch 39: loss=0.8692
Epoch 40: loss=0.8363
Epoch 41: loss=0.8697
Epoch 42: loss=0.8469
Epoch 43: loss=0.8418
Epoch 44: loss=0.8530
Epoch 45: loss=0.852

In [6]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

model.eval()
preds, trues = [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x, batch.edge_index, batch.batch)
        preds.extend((out.view(-1) > 0.5).int().tolist())
        trues.extend(batch.y.int().tolist())

print(f"Accuracy: {accuracy_score(trues, preds):.3f}")
print(f"F1 Score: {f1_score(trues, preds):.3f}")
print(f"Confusion Matrix:\n{confusion_matrix(trues, preds)}")

Accuracy: 0.812
F1 Score: 0.462
Confusion Matrix:
[[7154 1422]
 [ 412  789]]


In [7]:
torch.save(model.state_dict(), 'causal_gnn_weighted.pt')

In [8]:
def extract_causal_rule(model, data):
    model.eval()
    with torch.no_grad():
        x = F.relu(model.conv1(data.x, data.edge_index))
        x = F.relu(model.conv2(x, data.edge_index))
        pooled = x.mean(dim=0, keepdim=True)
        prob = torch.sigmoid(model.lin(pooled)).item()

    v0 = (data.x[0][-2]**2 + data.x[0][-1]**2).sqrt().item()
    v1 = (data.x[1][-2]**2 + data.x[1][-1]**2).sqrt().item()
    dominant = 0 if v0 > v1 else 1
    other = 1 - dominant

    true_label = "EXITED" if data.y.item() == 1 else "STAYED"

    if prob > 0.5:
        rule = f"Object {other} predicted to EXIT (confidence {prob:.2f}) — momentum from Object {dominant} exceeded stability threshold"
    else:
        rule = f"Object {other} predicted to STAY (confidence {1-prob:.2f}) — insufficient momentum transfer"

    return f"{rule} | Ground truth: {true_label}"

for i in range(10):
    print(extract_causal_rule(model, test_dataset[i]))

Object 1 predicted to STAY (confidence 0.80) — insufficient momentum transfer | Ground truth: EXITED
Object 1 predicted to EXIT (confidence 0.87) — momentum from Object 0 exceeded stability threshold | Ground truth: EXITED
Object 1 predicted to EXIT (confidence 0.54) — momentum from Object 0 exceeded stability threshold | Ground truth: EXITED
Object 0 predicted to EXIT (confidence 0.99) — momentum from Object 1 exceeded stability threshold | Ground truth: EXITED
Object 0 predicted to EXIT (confidence 0.89) — momentum from Object 1 exceeded stability threshold | Ground truth: EXITED
Object 1 predicted to EXIT (confidence 0.97) — momentum from Object 0 exceeded stability threshold | Ground truth: EXITED
Object 1 predicted to EXIT (confidence 0.55) — momentum from Object 0 exceeded stability threshold | Ground truth: EXITED
Object 1 predicted to EXIT (confidence 0.95) — momentum from Object 0 exceeded stability threshold | Ground truth: EXITED
Object 0 predicted to EXIT (confidence 0.89) 